# VAST-LoRA: Week 1 Competitor Board on Kaggle T4x2

This notebook clones a pinned VAST-LoRA commit, runs the reproducible in-house 3B baselines, and places the results next to the Week 1 literature targets.

Reproduced methods in this notebook:

- `raw`: naive asynchronous LoRA innovation aggregation.
- `fedex`: FedEx-LoRA-style exact innovation aggregation in the matched async simulator.
- `freshness`: whole-update freshness decay.
- `fedrot`: FedRot-LoRA-style Procrustes factor alignment before factor-space aggregation.
- `vast`: residual-only freshness after temporal projection.
- `mtip`: projection-only aggressive ablation.

Reference-only Week 1 opponents:

- GLoRA, FedRot-LoRA, FLoRG, FedEx-LoRA, SDFLoRA, FSLoRA, AlignFed.

Important: `fedex` and `fedrot` are matched-simulator ports, while the remaining paper rows are still reference-only. The board is designed to answer whether the current result is strong enough to claim a breakthrough without overclaiming against incompatible paper numbers.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time

REPO_URL = "https://github.com/TrgPhan/VASTLoRA.git"
REPO_REF = "b977f92d4226d407af904551d44096285bb7618c"
RUN_MODE = "full"  # Use "pilot" for one hard-regime seed.

WORK_ROOT = Path("/kaggle/working")
REPO_DIR = WORK_ROOT / "VASTLoRA-week1-board-run"
RESULT_DIR = WORK_ROOT / "vastlora-week1-competitor-results"
assert RUN_MODE in {"pilot", "full"}
print({"repo_ref": REPO_REF, "run_mode": RUN_MODE})

## 1. Clone and install

In [ ]:
if REPO_DIR.exists():
    assert REPO_DIR.parent == WORK_ROOT
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=REPO_DIR, check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO_DIR}[scale]"],
    check=True,
)
resolved_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
assert resolved_commit == REPO_REF
print("Installed commit:", resolved_commit)

## 2. Verify GPUs and cache data/model

In [ ]:
import torch

subprocess.run(["nvidia-smi"], check=True)
gpu_count = torch.cuda.device_count()
assert gpu_count >= 2, f"Expected Kaggle T4x2, found {gpu_count} CUDA device(s)"
gpu_names = [torch.cuda.get_device_name(index) for index in range(gpu_count)]
print("CUDA devices:", gpu_names)

In [ ]:
from datasets import load_dataset
from huggingface_hub import snapshot_download

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
snapshot_download(MODEL_NAME)
load_dataset("nyu-mll/glue", "sst2")
print("Model and SST-2 are cached.")

## 3. Prepare the reproducible baseline matrix

In [ ]:
BASE_CONFIG_PATH = REPO_DIR / "configs/kaggle_3b_slice_matrix.json"
RUNNER = REPO_DIR / "scripts/run_kaggle_3b.py"
SLICE_SUMMARY_SCRIPT = REPO_DIR / "scripts/summarize_kaggle_3b_slices.py"
BOARD_SCRIPT = REPO_DIR / "scripts/build_week1_competitor_board.py"
TARGETS = REPO_DIR / "configs/week1_competitor_targets.json"

base_config = json.loads(BASE_CONFIG_PATH.read_text(encoding="utf-8"))
regime_specs = {item["name"]: item for item in base_config["slice_matrix"]["regimes"]}

if RUN_MODE == "pilot":
    regime_names = ["noniid_high_staleness"]
    seeds = [3101]
else:
    regime_names = ["iid_homogeneous", "iid_heterogeneous", "noniid_high_staleness"]
    seeds = base_config["experiment"]["seeds"]
methods = ["raw", "fedex", "freshness", "fedrot", "vast", "mtip"]

if RESULT_DIR.exists():
    assert RESULT_DIR.parent == WORK_ROOT
    shutil.rmtree(RESULT_DIR)
RESULT_DIR.mkdir(parents=True)
CONFIG_DIR = RESULT_DIR / "configs"
LOG_DIR = RESULT_DIR / "logs"
CONFIG_DIR.mkdir()
LOG_DIR.mkdir()

def write_regime_config(regime_name):
    spec = regime_specs[regime_name]
    config = json.loads(json.dumps(base_config))
    config["output_dir"] = str(RESULT_DIR)
    config["experiment"]["partition_mode"] = spec["partition_mode"]
    config["experiment"]["client_ranks"] = spec["client_ranks"]
    config["experiment"]["methods"] = methods
    if RUN_MODE == "pilot":
        config["experiment"]["collected_returns"] = 16
        config["dataset"]["eval_examples"] = 256
    path = CONFIG_DIR / f"{regime_name}.json"
    path.write_text(json.dumps(config, indent=2), encoding="utf-8")
    return path

regime_configs = {name: write_regime_config(name) for name in regime_names}
for regime_name, config_path in regime_configs.items():
    for method in methods:
        subprocess.run(
            [sys.executable, str(RUNNER), "--config", str(config_path), "--method", method, "--dry-run"],
            cwd=REPO_DIR,
            check=True,
        )

## 4. Run reproduced methods

Every run uses the same model, dataset budget, client partition, and async trace for a given regime/seed. Full mode launches 54 jobs: 3 regimes x 3 seeds x 6 methods. Jobs are launched in pairs so each process owns one T4.

In [ ]:
jobs = [(regime, method, seed) for regime in regime_names for seed in seeds for method in methods]

def launch_job(regime, method, seed, gpu):
    variant = f"{regime}_{method}"
    log_path = LOG_DIR / f"{variant}_seed{seed}.log"
    log_handle = log_path.open("w", encoding="utf-8")
    env = os.environ.copy()
    env.update({
        "CUDA_VISIBLE_DEVICES": str(gpu),
        "PYTHONUNBUFFERED": "1",
        "TOKENIZERS_PARALLELISM": "false",
    })
    command = [
        sys.executable, str(RUNNER),
        "--config", str(regime_configs[regime]),
        "--method", method,
        "--variant", variant,
        "--seed", str(seed),
        "--output-dir", str(RESULT_DIR),
    ]
    process = subprocess.Popen(command, cwd=REPO_DIR, env=env, stdout=log_handle, stderr=subprocess.STDOUT)
    return process, log_handle, log_path

started = time.perf_counter()
for wave_start in range(0, len(jobs), 2):
    wave = jobs[wave_start:wave_start + 2]
    running = [launch_job(regime, method, seed, gpu) for gpu, (regime, method, seed) in enumerate(wave)]
    print("Started:", wave)
    while any(process.poll() is None for process, _, _ in running):
        time.sleep(30)
        active = [item for item, run in zip(wave, running) if run[0].poll() is None]
        print("  still running:", active)
    for (regime, method, seed), (process, handle, log_path) in zip(wave, running):
        handle.close()
        tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-20:]
        print(f"\n--- {regime}/{method} seed={seed}, exit={process.returncode} ---")
        print("\n".join(tail))
        if process.returncode != 0:
            raise RuntimeError(f"{regime}/{method} seed={seed} failed; inspect {log_path}")

wall_minutes = (time.perf_counter() - started) / 60
print(f"All {len(jobs)} jobs completed in {wall_minutes:.1f} minutes.")

## 5. Summarize slices and build the Week 1 competitor board

In [ ]:
subprocess.run(
    [sys.executable, str(SLICE_SUMMARY_SCRIPT), "--input-dir", str(RESULT_DIR), "--target-method", "vast"],
    cwd=REPO_DIR,
    check=True,
)
BOARD_DIR = RESULT_DIR / "week1_competitor_board"
subprocess.run(
    [
        sys.executable, str(BOARD_SCRIPT),
        "--targets", str(TARGETS),
        "--our-summary-dir", str(RESULT_DIR / "slice_summary"),
        "--output-dir", str(BOARD_DIR),
    ],
    cwd=REPO_DIR,
    check=True,
)

import pandas as pd
from IPython.display import Markdown, display

slice_summary = RESULT_DIR / "slice_summary"
method_summary = pd.read_csv(slice_summary / "method_summary.csv")
regime_summary = pd.read_csv(slice_summary / "regime_summary.csv")
paired = pd.read_csv(slice_summary / "paired_comparisons.csv")
board = pd.read_csv(BOARD_DIR / "week1_competitor_board.csv")
verdict = json.loads((BOARD_DIR / "competitor_verdict.json").read_text(encoding="utf-8"))

display(Markdown((BOARD_DIR / "competitor_verdict.md").read_text(encoding="utf-8")))
display(Markdown("### Reproduced regime summary"))
display(regime_summary)
display(Markdown("### Reproduced paired comparisons vs Freshness"))
display(paired)
display(Markdown("### Week 1 competitor board"))
display(board[[
    "row_type", "framework", "dataset", "metric", "score", "reported_score",
    "model_or_backbone", "setting", "reproduction_status", "comparable_fairly",
]])

## 6. Plot reproduced methods and reference targets

In [ ]:
import matplotlib.pyplot as plt

acc_rows = board[board["metric"].isin(["mean_accuracy", "average_accuracy", "accuracy", "average_score"])]
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
ours = acc_rows[acc_rows["row_type"].eq("our_reproduced_accuracy")]
refs = acc_rows[acc_rows["row_type"].eq("literature_reference")]
axes[0].barh(ours["framework"] + " / " + ours["setting"], 100 * ours["score"])
axes[0].set_title("Reproduced Kaggle runs")
axes[0].set_xlabel("Accuracy (%)")
axes[1].barh(refs["framework"] + " / " + refs["dataset"], 100 * refs["score"])
axes[1].set_title("Week 1 reference-only targets")
axes[1].set_xlabel("Reported score (%)")
for axis in axes:
    axis.grid(axis="x", alpha=0.25)
fig.tight_layout()
plt.show()

## 7. Reproducibility record and archive

In [ ]:
from IPython.display import FileLink

record = {
    "git_commit": resolved_commit,
    "run_mode": RUN_MODE,
    "model": MODEL_NAME,
    "gpus": gpu_names,
    "regimes": regime_names,
    "methods": methods,
    "seeds": seeds,
    "wall_minutes": wall_minutes,
    "competitor_verdict": verdict,
}
(RESULT_DIR / "run_record.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
archive = shutil.make_archive(str(WORK_ROOT / "vastlora-week1-competitor-results"), "zip", RESULT_DIR)
display(record)
display(FileLink(archive))

### Interpretation rule

- `reproduced_in_this_notebook`: same runner, same model, same FL simulator, fair within this notebook.
- `reference_only_public_code`: paper has public-code status in Week 1 notes, but that exact paper setting has not been rerun here.
- `reference_only`: literature number only, not a fair baseline in this notebook.

`fedex` and `fedrot` provide the first matched external ports for this simulator. A broad breakthrough claim still requires VAST to beat or remain non-inferior to those ports on accuracy and decision calibration, plus reliability wins on NLL/Brier/ECE/staleness slices. This notebook should not be used to overclaim against paper numbers produced with different models, tasks, clients, and evaluation protocols.